# 🧮 Math Explorer

**Welcome!** This notebook is your personal math workspace.

Here's how it works:
1. **Run the Setup cell** (▶ button) — takes about 30 seconds the first time
2. **Read the assignment** below
3. **Use the interactive widget** to explore the problem — try things, make mistakes, experiment
4. **Get a hint** any time you're stuck
5. **Write your reasoning** in the My Work section
6. **Submit** when you're done

> 💡 **Tip:** You can't break anything. Try different approaches freely — that's how you learn.


In [ ]:
# @title ⚙️ Setup — Run this first (do not edit) { display-mode: "form" }
# @markdown Downloads the math widget and installs required packages.
# @markdown **Run once at the start of every session.**

import sys, subprocess

print("Installing packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "anthropic", "comm", "sympy", "matplotlib"],
    check=True, capture_output=True
)

print("Downloading widget files...")
import urllib.request
BASE = "https://raw.githubusercontent.com/muftring/math-expression-widget/master/"
for fname in ["math_widget.html", "notebook_integration.py"]:
    urllib.request.urlretrieve(BASE + fname, fname)

print("\u2713 Setup complete — ready to go!")


In [ ]:
# @title 📚 Assignment Configuration (Teachers: edit this cell) { display-mode: "form" }

ASSIGNMENT_TITLE  = "Factoring Quadratic Equations"  # @param {type:"string"}
GRADE_LEVEL       = "9th Grade Algebra"               # @param {type:"string"}
TOPIC             = "Factoring"                       # @param ["Factoring", "Solving Equations", "Graphing", "Inequalities", "Geometry", "Systems of Equations", "Other"]

# The LaTeX expression that pre-loads into the widget
STARTER_EXPRESSION = "x^2 - 5x + 6"  # @param {type:"string"}

# What the student should do
INSTRUCTIONS = "Factor the quadratic expression completely. Show your reasoning step by step."  # @param {type:"string"}

# The learning goal shown to the student
LEARNING_OBJECTIVE = "I can factor a quadratic trinomial by finding two numbers that multiply to c and add to b."  # @param {type:"string"}

# Turn hints on or off for this assignment
HINTS_ALLOWED = True  # @param {type:"boolean"}


In [ ]:
# Display the assignment — run after editing the configuration above
from IPython.display import display, Markdown

display(Markdown(f"""
---
## \U0001f4dd {ASSIGNMENT_TITLE}
*{GRADE_LEVEL}*

### The Problem
{INSTRUCTIONS}

$$
{STARTER_EXPRESSION}
$$

### Learning Goal
> {LEARNING_OBJECTIVE}

---
"""))


## 🔢 Step 1 — Explore with the Math Widget

Use the widget below to work through the problem. Try things — expand, factor, solve, plot.
Everything you do is saved automatically.

**When you're ready to move on**, scroll down to Step 2.


In [ ]:
# Initialize the widget backend — run this before displaying the widget
from notebook_integration import setup_math_widget
setup_math_widget()


In [ ]:
# Display the interactive math widget
import json
from IPython.display import display, HTML

def display_widget(starter=""):
    with open("math_widget.html", "r") as f:
        html = f.read()

    # Inject a small script to pre-populate the expression input
    if starter:
        preload = f"""
<script>
(function() {{
  var starter = {json.dumps(starter)};
  function trySet() {{
    var el = document.getElementById('latex-input');
    if (el) {{ el.value = starter; }}
    else    {{ setTimeout(trySet, 150); }}
  }}
  trySet();
}})();
</script>"""
        html = html.replace('</body>', preload + '\n</body>')

    display(HTML(html))

display_widget(STARTER_EXPRESSION)


## 💡 Step 2 — Get a Hint (if you need one)

Stuck? Describe where you are, then click **Get a Hint**.
The hint will ask you a question to help you think — it won't give you the answer.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown
import anthropic

def _get_api_key():
    """Retrieve API key from Colab Secrets, env var, or return None."""
    try:
        from google.colab import userdata
        return userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        pass
    import os
    return os.environ.get('ANTHROPIC_API_KEY')

def _request_hint(question=""):
    from notebook_integration import received_expressions

    api_key = _get_api_key()
    if not api_key:
        display(Markdown(
            "> \u26a0\ufe0f **Hints unavailable.**  \n"
            "> Ask your teacher to enable hints for this assignment."
        ))
        return

    # Summarise what the student has tried in the widget
    if received_expressions:
        tried = "\n".join(f"- ${e}$" for e in received_expressions[-6:])
        work_summary = f"The student has worked with these expressions:\n{tried}"
    else:
        work_summary = "The student has not entered any expressions yet."

    client = anthropic.Anthropic(api_key=api_key)

    try:
        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=300,
            system=(
                f"You are a patient, encouraging math tutor for a {GRADE_LEVEL} student.\n\n"
                f"Assignment: {ASSIGNMENT_TITLE}\n"
                f"Topic: {TOPIC}\n"
                f"Learning objective: {LEARNING_OBJECTIVE}\n\n"
                "Rules:\n"
                "- NEVER give the answer or solve the problem for the student.\n"
                "- Ask exactly ONE guiding question that moves them one step forward.\n"
                "- If they have made progress, acknowledge it specifically and warmly.\n"
                "- Use simple, age-appropriate, encouraging language.\n"
                "- Keep your entire response under 4 sentences."
            ),
            messages=[{
                "role": "user",
                "content": (
                    f"Problem: ${STARTER_EXPRESSION}$\n\n"
                    f"{work_summary}\n\n"
                    f"Student says: {question if question else 'I need a hint to move forward.'}\n\n"
                    "Give me one helpful hint."
                )
            }]
        )
        hint = response.content[0].text
        display(Markdown(
            f"\n---\n### \U0001f4a1 Hint\n\n{hint}\n\n"
            "---\n*Keep going \u2014 you've got this!* \U0001f31f"
        ))
    except Exception as exc:
        display(Markdown(f"> \u274c Could not get hint: `{exc}`"))


if HINTS_ALLOWED:
    _question_box = widgets.Textarea(
        placeholder="Describe where you are stuck, or what you have already tried...",
        layout=widgets.Layout(width='100%', height='80px')
    )
    _hint_btn = widgets.Button(
        description='Get a Hint',
        icon='lightbulb',
        style={'button_color': '#a6e3a1'},
        layout=widgets.Layout(width='160px', margin='6px 0')
    )
    _hint_out = widgets.Output()

    def _on_hint_click(b):
        _hint_btn.disabled = True
        _hint_btn.description = 'Thinking...'
        with _hint_out:
            _hint_out.clear_output(wait=True)
            _request_hint(_question_box.value)
        _hint_btn.disabled = False
        _hint_btn.description = 'Get a Hint'

    _hint_btn.on_click(_on_hint_click)

    display(widgets.VBox([
        _question_box,
        _hint_btn,
        _hint_out
    ]))
else:
    display(Markdown("> \u2139\ufe0f Hints are not enabled for this assignment."))


## 📝 Step 3 — My Work and Reasoning

*Double-click this cell to edit it. Write out your thinking below.*

---

**My approach — what strategy did I use?**

*(write here)*

**My steps:**

1. 
2. 
3. 

**My answer:**

*(write here)*

**How I checked my answer:**

*(write here)*


## 🤔 Step 4 — Reflection

*Double-click to edit. Complete this after finishing the problem.*

---

**What was the hardest part of this problem?**

*(write here)*

**What would you do differently next time?**

*(write here)*

**Confidence level** *(circle one)*:  
⬜ Still confused &nbsp;&nbsp; ⬜ Getting there &nbsp;&nbsp; ⬜ Got it! &nbsp;&nbsp; ⬜ I could teach this


In [ ]:
# @title 📤 Step 5 — Prepare Submission { display-mode: "form" }
# @markdown Run this cell to review your work, then submit via Google Classroom.

from notebook_integration import received_expressions
from IPython.display import display, Markdown
from datetime import datetime

lines = [
    "## \U0001f4cb Submission Summary",
    "",
    f"**Assignment:** {ASSIGNMENT_TITLE}",
    f"**Problem:** ${STARTER_EXPRESSION}$",
    f"**Submitted:** {datetime.now().strftime('%B %d, %Y at %I:%M %p')}",
    "",
    f"**Expressions explored ({len(received_expressions)} total):**",
]

if received_expressions:
    for expr in received_expressions:
        lines.append(f"- ${expr}$")
else:
    lines.append("- *(no expressions recorded — make sure you used the widget above)*")

lines += [
    "",
    "---",
    "**\u2705 Ready to submit?**",
    "1. Make sure you've filled in **My Work** and **Reflection** above",
    "2. Go to **File → Save** (or press Ctrl+S)",
    "3. Submit this notebook in **Google Classroom**",
]

display(Markdown("\n".join(lines)))
